# Ark+ (Swin-L) baseline on CheXpert — Colab

**Before running:**
1. `Runtime > Change runtime type` → GPU. T4 works fine for the current config (see below); for the real 768px Ark+ baseline later, use the fastest available (H100 > A100 > L4 > T4) — needs Colab Pro/Pro+ or pay-as-you-go compute units.
2. `USE_ARK_WEIGHTS` (cell 12) is now **`True`** — the real Ark+ checkpoint (`Ark6_swinLarge768_ep50.pth.tar`) was verified 2026-08-03 (backbone loads clean; only window-geometry buffers + head differ, as expected for a 768px->320px transfer) and is on Drive. Set back to `False` only if you want the random-init control instead — cell 12 checks Drive first, then falls back to an upload dialog if not found there.

Cell 14 has a `NUM_CLASSES` toggle: **14** (current default) trains the model head on all 14 CheXpert labels but only reports AUROC/etc. on the 5 competition classes — the original Ark+ paper's protocol. **5** trains the head on just the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) directly; section 2's patch 3 makes that possible. Built on the official [jlianglab/Ark](https://github.com/jlianglab/Ark) finetuning code.

**On speed:** the repo's training loop originally created a `NativeScaler` but never actually wrapped the forward pass in `torch.autocast` — no mixed-precision benefit despite appearances. Section 2's patch 6 fixes this for real (`bfloat16` autocast — native on Ampere/A100 and even stronger on Hopper/H100, no gradient scaling needed unlike fp16). With `USE_ARK_WEIGHTS=False`, cell 14 auto-switches to a cheap 224px config (`BATCH_SIZE=32`); flipping to `True` switches to the heavy 768px config Ark+ actually needs (`BATCH_SIZE=4`, conservative but proven to work on A100 — tune down further if you hit an OOM).

**On accuracy:** cell 14 also has three toggles (`USE_PER_CLASS_UNCERTAIN_LABEL`, `USE_CLASS_WEIGHTED_LOSS`, `USE_COOCCURRENCE_LOSS`, all `True` by default) for data-driven improvements built from gaps observed in the initial baseline run plus recent chest X-ray literature — see section 2 (patches 10-12) and section 4 for details. Set any to `False` to A/B against the plain baseline.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Kaggle credentials + dataset download

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 2. Clone Ark+ repo, patch it, and install dependencies
The original repo pins 2021-era package versions. `timm==0.5.4` and `albumentations==0.4.3` are kept exact — the repo's README warns newer timm versions changed the Swin Transformer architecture enough to break pretrained-weight loading, and the dataloader uses transform names albumentations removed later. We patch one `np.bool` alias (removed in numpy>=1.24) instead of downgrading numpy globally, to avoid conflicting with Colab's preinstalled stack.

Patches applied here (all verified locally first):
- `models.py` imports `UperNet_swin` from a module that isn't actually included in this repo folder — a dead, unused import that crashes on load. Removed.
- `dataloader.py`'s `CheXpert` dataset always reads all 14 label columns. We add an optional `class_indices` argument so it can select just the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) in the right order — this is what powers the `NUM_CLASSES=5` option in cell 12's toggle. With `NUM_CLASSES=14` (the current default), `class_indices` is left `None` and the dataset falls back to its original all-14-columns behavior.
- Three data-driven improvements (patches 10-12), motivated by gaps observed in the initial baseline run and recent chest X-ray literature — each independently toggleable in cell 14:
  - **Per-pathology uncertain-label strategy** (patch 10): `dataloader.py`'s `uncertain_label` can now be a list (one strategy per class) instead of only a single global string — the original CheXpert paper's own U-Ones/U-Zeros ablation found the best strategy is pathology-dependent.
  - **Class-weighted loss** (patches 11/12): a `pos_weight` on the BCE loss, computed from the real training-set class balance, to counter the imbalance behind low-AP classes like Consolidation.
  - **Co-occurrence-aware regularization** (patches 11/12): a small penalty (`utils.CoOccurrenceBCELoss`) that nudges the classifier head's weight vectors for commonly co-occurring pathologies to point in more similar directions — a simplified, batch-size-independent take on hierarchical/dependency-aware losses in recent literature (e.g. HBCE, arXiv:2502.03591).

This cell is safe to re-run (e.g. after a kernel restart) — the clone and patches are skipped/no-op if already present.

In [ ]:
import os, sys

# Install the same dependencies the training code needs (this doesn't change: copying our own
# code below doesn't install third-party packages).
!{sys.executable} -m pip install -q --break-system-packages timm==0.5.4 albumentations==0.4.3 opencv-python-headless scikit-image scikit-learn \
    simpleitk scipy pydicom yacs einops transformers pretrainedmodels imgaug pillow tqdm

# imgaug (a transitive dep of albumentations==0.4.3, old and unmaintained) has a np.sctypes
# reference (removed in numpy 2.0) in parameters.py, plus a dead `from scipy import misc` import
# (removed in modern scipy) in the same file. Both crash on *import*, so we can't `import imgaug`
# first to locate the file -- that import is exactly what crashes. Using importlib.util.find_spec
# instead, which locates the module without executing it.
import importlib.util, os as _os
_ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
_ia_params = _os.path.join(_ia_dir, 'parameters.py')
_s = open(_ia_params).read()
_before = _s
_s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
_s = _s.replace('from scipy import misc\n', '')
if _s != _before:
    open(_ia_params, 'w').write(_s)
    print('imgaug patched:', _ia_params)
else:
    print('imgaug already patched (or patterns not found), skipping:', _ia_params)
import imgaug
print('imgaug import OK, version', imgaug.__version__)

# The actual CheXpert training code (dataloader/trainer/engine/models) is pulled straight from
# Drive instead of re-deriving ~15 sed/python patches against a live jlianglab/Ark clone every
# run. This copy already has everything: base compatibility fixes, all 6 models wired in, and
# the full frozen protocol (320px support for every architecture, patient-level held-out test
# split, stratified ablation subset, ECE metric, configurable AMP dtype, per-trial seeding) --
# kept in sync with the canonical copy at github.com/fei7fei/chexpert-baselines. Every notebook
# reads from this exact same folder, so the underlying code is guaranteed byte-identical across
# all 6 models -- required for the frozen-protocol comparison to be valid.
from google.colab import drive
drive.mount('/content/drive')

_DRIVE_CODE_SRC = '/content/drive/MyDrive/Project/models/Repos/ark_plus_repo/Ark_Plus/Finetuning'
if not os.path.exists('/content/chexpert_code'):
    import shutil
    shutil.copytree(_DRIVE_CODE_SRC, '/content/chexpert_code')
%cd /content/chexpert_code
print('Loaded pre-patched CheXpert training code (frozen protocol v1) from Drive.')

In [ ]:
import torch, timm
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))
print('timm', timm.__version__)
from timm.models.swin_transformer import SwinTransformer
m = SwinTransformer(num_classes=5, img_size=768, patch_size=4, window_size=12, embed_dim=192,
                     depths=(2,2,18,2), num_heads=(6,12,24,48))
print('Swin-L (768, window12) built OK, params:', sum(p.numel() for p in m.parameters()))
del m

## 3. Mount Google Drive for checkpoint persistence
`/content` is ephemeral — if the runtime disconnects or gets recycled (e.g. Colab Compute Units running out), everything in it is gone, checkpoints included. This mounts Drive and points training's save location there instead, so progress survives even a full runtime loss. Re-running cell 14 after a restart will then automatically resume from the last saved epoch (via `args.resume`) instead of starting over. Mounted first (before the checkpoint step below) so that step can check Drive for an already-uploaded Ark+ checkpoint instead of always requiring the upload dialog.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/ark_plus_chexpert_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints and results will be saved to:', DRIVE_CKPT_DIR)

## 3b. Ark+ pretrained checkpoint
`USE_ARK_WEIGHTS = True` (set below) uses the real Ark+ checkpoint (`Ark6_swinLarge768_ep50.pth.tar`, on Drive as of 2026-08-03, verified: backbone loads clean, only window-geometry buffers + head differ). Set to `False` to train Swin-L from random init instead (the SwinL control notebook's config).

Checks Drive first (`DRIVE_CKPT_DIR/Ark6_swinLarge768_ep50.pth.tar`) before falling back to the upload dialog — upload once, and it's there for every future session without re-uploading. Also more robust than relying on the upload dialog alone: `files.upload()` is the same class of browser-dependent Colab widget (`google.colab.userdata`) that silently failed earlier this session when connected through the VS Code↔Colab bridge rather than the browser.

In [ ]:
import os, shutil

USE_ARK_WEIGHTS = True  # real Ark+ checkpoint verified 2026-08-03 -- backbone loads clean, only window-geometry buffers + head differ, as expected

CKPT_FILENAME = 'Ark6_swinLarge768_ep50.pth.tar'
DRIVE_CKPT_PATH = os.path.join(DRIVE_CKPT_DIR, CKPT_FILENAME)
CKPT_PATH = f'/content/chexpert_code/{CKPT_FILENAME}'

if not USE_ARK_WEIGHTS:
    CKPT_PATH = None
    print('No checkpoint — will train Swin-L from random init.')
elif os.path.exists(DRIVE_CKPT_PATH):
    shutil.copy(DRIVE_CKPT_PATH, CKPT_PATH)
    print('Found checkpoint on Drive, using it directly:', CKPT_PATH)
else:
    print('Checkpoint not found on Drive -- falling back to upload dialog.')
    print('(If this hangs or fails silently through the VS Code<->Colab bridge, try running')
    print('this same cell from browser Colab instead, then rerun it here once uploaded.)')
    from google.colab import files
    uploaded = files.upload()  # select Ark6_swinLarge768_ep50.pth.tar
    ckpt_name = list(uploaded.keys())[0]
    os.rename(ckpt_name, CKPT_PATH)
    shutil.copy(CKPT_PATH, DRIVE_CKPT_PATH)  # so future sessions skip the upload entirely
    print('Uploaded and cached to Drive for next time:', DRIVE_CKPT_PATH)
    print('Using checkpoint:', CKPT_PATH)

## 4. Run finetuning on the CheXpert benchmark
Trains Swin-L per the `NUM_CLASSES` toggle set in the cell below, reporting AUROC/MCC/AP/F1 on the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) each epoch either way. Adjust `BATCH_SIZE`/`EPOCHS`/`PATIENCE` there for your GPU and time budget.

Three toggles just below `NUM_CLASSES` control the data-driven improvements from section 2 (patches 10-12) — all default `True`, set any to `False` to A/B against the plain baseline:
- `USE_PER_CLASS_UNCERTAIN_LABEL` — per-pathology uncertain-label strategy instead of one global `"LSR-Ones"`.
- `USE_CLASS_WEIGHTED_LOSS` — `pos_weight` on the BCE loss, computed from the actual training-set class balance.
- `USE_COOCCURRENCE_LOSS` — co-occurrence-aware regularization on the classifier head's weights.

`args.exp_name` (and so the checkpoint directory) encodes which of these are active, so flipping a toggle starts a fresh run instead of accidentally resuming into a mismatched one.

In [ ]:
%cd /content/chexpert_code
import sys, os
sys.path.insert(0, '.')

_old_argv = sys.argv
sys.argv = ['main_classification.py']
from main_classification import get_args_parser
args = get_args_parser()
sys.argv = _old_argv

# ===== Frozen protocol v1 -- identical across all 6 models, fix this before running anything =====
INPUT_SIZE = 320                    # native CheXpert-small short side -- avoids gratuitous up/downsampling
UNCERTAIN_LABEL = "Zeros"           # U-Zeros for main results; flip to "Ones" for the appendix sensitivity check
EPOCHS = 30
SEEDS_SCREENING = [13]              # 1 seed: cheap ranking pass on the 20% ablation subset
SEEDS_HEADLINE = [13, 42]     # 3 seeds: confirmation pass on the full training pool

args.model_name = "swin_large"
args.init = "ark" if USE_ARK_WEIGHTS else "random"
args.pretrained_weights = CKPT_PATH
args.scale_up = True  # window_size changes 12 (768px pretrain) -> 5 (320px protocol); needed for the shape-mismatch filter in load_pretrained_weights
args.input_size = INPUT_SIZE
args.img_size = INPUT_SIZE
args.num_class = 14   # train the head on all 14 CheXpert labels; report only on the 5 competition
                       # classes via test_diseases below (original Ark+ protocol convention)
args.data_set = "CheXpert"
args.data_dir = "/content/data"
args.key = "teacher"
args.batch_size = 64   # "largest that fits" per protocol -- reduce if you hit OOM at 320px,
                                  # or wire in gradient checkpointing (not done here)
args.epochs = EPOCHS
args.patience = EPOCHS           # no early stopping -- every model trains the full 50 epochs so run
                                  # lengths stay comparable across the whole comparison
args.opt = "adamw"
args.lr = 5e-5
args.weight_decay = 0.05
args.sched = "cosine"
args.warmup_epochs = 1
args.cooldown_epochs = 0
args.uncertain_label = UNCERTAIN_LABEL
args.test_every_epoch = True
args.save_every_epoch_checkpoint = True
args.amp_dtype = "fp16"
args.device = "cuda"
args.resume = True
args.workers = 8  # bumped from 2: a rock-steady per-batch time with zero jitter is the signature of a CPU-bound dataloader, not a slow GPU

diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
test_diseases_5 = [diseases_14.index(c) for c in competition_5]

from dataloader import CheXpert, build_transform_classification, build_protocol_splits, filter_csv_by_patients
from engine import classification_engine

# Patient-level held-out test (carved from train) + stratified 20% ablation subset -- computed
# once and cached to a *shared* Drive folder (not this model's own checkpoint folder) so every
# one of the 6 models' notebooks reads the exact same patient partition, in whatever order they
# happen to run. That's what makes the cross-model comparison valid rather than 6 separate
# anecdotes on 6 different splits.
PROTOCOL_SHARED_DIR = '/content/drive/MyDrive/chexpert_protocol_v1'
os.makedirs(PROTOCOL_SHARED_DIR, exist_ok=True)

train_csv = os.path.join(args.data_dir, "train.csv")
splits = build_protocol_splits(train_csv, competition_5, cache_path=os.path.join(PROTOCOL_SHARED_DIR, "protocol_split.csv"))

def _cached_filter(name, patient_set):
    p = os.path.join(PROTOCOL_SHARED_DIR, f"{name}.csv")
    if not os.path.exists(p):
        filter_csv_by_patients(train_csv, patient_set, p)
    return p

train_pool_csv = _cached_filter("train_pool", splits["train_pool"])
held_out_test_csv = _cached_filter("held_out_test", splits["held_out_test"])
ablation_subset_csv = _cached_filter("ablation_subset", splits["ablation_subset"])
official_val_csv = os.path.join(args.data_dir, "valid.csv")

args.train_list = train_pool_csv
args.val_list = official_val_csv     # the official 234-study set -- used for per-epoch threshold
                                      # tuning/checkpoint selection only, never for final reporting
args.test_list = held_out_test_csv   # final reporting -- held out from train, never seen in training

def make_ds(mode, file_path):
    return CheXpert(images_path=args.data_dir, file_path=file_path,
        augment=build_transform_classification(normalize=args.normalization, mode=mode, crop_size=args.input_size, resize=args.img_size),
        uncertain_label=args.uncertain_label, unknown_label=args.unknown_label)

dataset_val = make_ds("valid", args.val_list)
dataset_test = make_ds("test", args.test_list)
print(f"val (official, threshold tuning): {len(dataset_val)}  |  test (held-out, final report): {len(dataset_test)}")

model_path = os.path.join(DRIVE_CKPT_DIR, "Models/Classification", args.data_set)
output_path = os.path.join(DRIVE_CKPT_DIR, "Outputs/Classification", args.data_set)

# ===== Tier 1: screening -- cheap, single seed, ranks variants on the 20% subset =====
print("\n=== TIER 1: screening (1 seed, 20% stratified subset) ===")
args.exp_name = "_protocol1_arkplus_screening"
args.num_trial = len(SEEDS_SCREENING)
args.start_index = 0
args.seed_list = SEEDS_SCREENING
dataset_train_screen = make_ds("train", ablation_subset_csv)
print(f"screening train size: {len(dataset_train_screen)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_screen, dataset_val, dataset_test, test_diseases=test_diseases_5)

# ===== Tier 2: headline -- 3 seeds, full training data. Report both tiers; the two-tier split =====
# ===== is itself part of what makes ~20 runs comparable rather than 20 anecdotes. =====
print("\n=== TIER 2: headline (3 seeds, full training pool) ===")
args.exp_name = "_protocol1_arkplus_headline"
args.num_trial = len(SEEDS_HEADLINE)
args.start_index = 0
args.seed_list = SEEDS_HEADLINE
dataset_train_full = make_ds("train", train_pool_csv)
print(f"headline train size: {len(dataset_train_full)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_full, dataset_val, dataset_test, test_diseases=test_diseases_5)

## 5. Results
Per-epoch AUROC prints during training above. Final summary (mean/std AUC, MCC, AP, F1 across the 5 classes) lands in `Outputs/Classification/CheXpert/*_results.txt`; checkpoints in `Models/Classification/CheXpert/` — both now inside `DRIVE_CKPT_DIR` on your Google Drive (see cell 12), so they survive a runtime loss. If cell 14 gets cut off mid-run, just re-run it — `args.resume=True` picks back up from the last saved epoch automatically.

The next cell shows the clean per-epoch table (Epoch, train_loss, val_loss, F1, AUC, recall, accuracy) as a DataFrame — patch 9 (cell 7) writes this to its own CSV alongside the verbose text log, rewritten after every epoch so it's readable even from a run that got interrupted partway through.

In [ ]:
!find "{DRIVE_CKPT_DIR}/Outputs" -type f
!echo ---
!cat "{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert"/*_results.txt

In [ ]:
import glob
import pandas as pd

_csvs = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv"))
if not _csvs:
    print("No epoch table found yet -- run cell 14 through at least one epoch first.")
else:
    _latest = _csvs[-1]
    print("Showing:", _latest)
    display(pd.read_csv(_latest))

## 6. Publication-style comparison table
Builds a `Method | Acc | Sen | F1 | AUC` table (the format common in chest X-ray classification papers) from every `*_results.txt` file found in `Outputs/Classification/CheXpert/` — one row per experiment (baseline, baseline+improvements, or any ablation you run by toggling cell 14's `USE_*` flags). "Sen" (Sensitivity) is the same quantity as `Recall` in the per-epoch table above; renamed here to match the convention in most published tables.

Note: with `args.num_trial=1` (the current default), there's only a single run per configuration, so there's no real cross-run standard deviation to report — the `(mean ± std)` style you see in papers comes from averaging several independently-seeded trials. Bump `args.num_trial` in cell 14 (e.g. to 3) and rerun if you want genuine ± error bars later; that costs proportionally more compute.

In [ ]:
import re, glob, os
import pandas as pd

_pattern = re.compile(
    r"aAUC = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aMCC = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aAP = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aF1 = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aRecall = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aAccuracy = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aECE = (-?[\d.]+)\((-?[\d.]+)\)"
)

def _tier_label(fname):
    # exp_name convention (see the config+training cell): _protocol1_<model>_screening /
    # _protocol1_<model>_headline. Two-tier discipline: screening is a cheap single-seed
    # ranking pass on the 20% stratified subset; headline is the confirmed 3-seed result on
    # the full training pool. Both are reported explicitly and separately below -- an
    # unlabeled/blended number is exactly what an unexplained-subset result looks like.
    base = os.path.basename(fname).replace("_results.txt", "")
    if base.endswith("_screening"):
        return "Screening (1 seed, 20% subset)"
    if base.endswith("_headline"):
        return "Headline (3 seeds, 100% data)"
    return base.lstrip("_").replace("_", " ")  # fallback for any other exp_name found here

rows = []
for f in sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert/*_results.txt")):
    text = open(f).read()
    m = _pattern.search(text)
    if not m:
        print("Could not parse (no 'Mean / STD over All trials' line found -- run may still be in progress):", f)
        continue
    auc, auc_std, mcc, mcc_std, ap, ap_std, f1, f1_std, recall, recall_std, acc, acc_std, ece, ece_std = map(float, m.groups())
    rows.append({
        "Tier": _tier_label(f),
        "Acc (%)": f"{acc*100:.1f} +/- {acc_std*100:.1f}",
        "Sen (%)": f"{recall*100:.1f} +/- {recall_std*100:.1f}",
        "F1 (%)": f"{f1*100:.1f} +/- {f1_std*100:.1f}",
        "AUC (%)": f"{auc*100:.1f} +/- {auc_std*100:.1f}",
        "ECE (%)": f"{ece*100:.1f} +/- {ece_std*100:.1f}",
    })

if not rows:
    print("No results files found yet -- run the config+training cell through completion first.")
else:
    df = pd.DataFrame(rows).set_index("Tier")
    print("Two-tier protocol results, reported separately as required: screening (single seed,")
    print("cheap ranking pass on the 20% subset) and headline (3-seed confirmation on the full")
    print("training pool). Screening's +/- is trivially 0.0 (n=1) -- only headline's spread is")
    print("meaningful; shown for both so the table format stays consistent.\n")
    display(df)

## 6. Gap #1 -- Calibration and per-label operating-point analysis

Post-hoc, ~0 GPU cost (Experiment Plan v3, SS5.2(b)) -- runs on cached/quickly-recomputed logits, no additional training. Answers: does large-scale pretraining produce a better-calibrated model, and does threshold policy close the accuracy/sensitivity gap that raw AUC hides?

**Also fixes a real leak found while building this**: `engine.py`'s test-time F1/Recall/Accuracy pick each class's Youden's-J threshold from the *test* set's own labels, then score that same set -- an optimistic bias, same category as the checkpoint-selection leak already fixed this session (`SESSION_NOTES.md` bug #1). AUC is unaffected (threshold-independent); the base pipeline's F1/Recall/Accuracy are, and shouldn't be used as headline numbers until this cell's val-tuned versions are available.

Requires cell 14 to have completed at least one full run (screening and/or headline) for the checkpoint + cached test predictions to exist -- safe to leave in place and re-run later as more experiments finish; it skips any run it can't find a checkpoint for.

**Not yet execution-verified end-to-end** (written 2026-08-03, before any full training run under the current 30-epoch/2-seed protocol had completed) -- every function it calls (`test_model`, `read_from_csv`, `meanECE`, `build_classification_model`) is reused directly from the existing, already-verified pipeline, but run this cell once real checkpoints exist and sanity-check the output before trusting it in the paper.

In [ ]:
## Gap #1: calibration + per-label operating-point analysis (post-hoc, no training needed)
##
## engine.py's test-time meanF1/meanRecall/meanAccuracy (see the "start testing" block) pick
## each class's Youden's-J-optimal threshold FROM the held-out test set's own labels, then
## immediately score that same set with it -- an optimistic leak, the same category of bug as
## the checkpoint-selection leak fixed earlier this session (SESSION_NOTES.md bug #1), just for
## thresholds instead of epochs. AUC is unaffected (threshold-independent); F1/Recall/Accuracy
## as currently reported by the base pipeline are not, and shouldn't be treated as final numbers.
##
## This cell fits temperature (calibration) and three threshold policies on VALIDATION only, then
## applies them ONCE to the held-out test set -- no leak, by construction. Per PROTOCOL.md this
## is Youden's J; the earlier gap-analysis plan doc used per-class F1-optimal instead -- both are
## reported here alongside the naive 0.5 default, so the spread across policies is itself part of
## the result rather than a hidden methodology choice (matches Experiment Plan v3 SS5.2(b)).
##
## Reports each seed individually, not pooled -- with 2 headline seeds, a mean+/-range would
## misrepresent a 2-point range as an error bar (Experiment Plan v3 SS6.3).

import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_curve, f1_score, recall_score, accuracy_score
import torch
from torch.utils.data import DataLoader

from models import build_classification_model
from trainer import test_model
from utils import read_from_csv, meanECE


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def _sigmoid(x):
    return 1 / (1 + np.exp(-x))


def fit_temperature(y_val, p_val):
    # Single scalar T (one per model, not per-class) minimizing NLL on validation -- standard
    # temperature scaling (Guo et al., ICML 2017). p_val is post-sigmoid (test_model's output),
    # so logits are recovered via inverse-sigmoid rather than re-deriving raw logits from scratch.
    logits = _logit(p_val)

    def nll(T):
        T = max(T, 1e-3)
        p = np.clip(_sigmoid(logits / T), 1e-7, 1 - 1e-7)
        return -np.mean(y_val * np.log(p) + (1 - y_val) * np.log(1 - p))

    res = minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded")
    return res.x


def per_label_thresholds(y_val, p_val, mode):
    thresholds = np.full(y_val.shape[1], 0.5)
    for i in range(y_val.shape[1]):
        if not np.any(y_val[:, i]):
            continue
        fpr, tpr, thr = roc_curve(y_val[:, i], p_val[:, i])
        if mode == "youden":
            thresholds[i] = thr[np.argmax(tpr - fpr)]
        elif mode == "f1":
            f1s = [f1_score(y_val[:, i], (p_val[:, i] > t).astype(int)) for t in thr]
            thresholds[i] = thr[int(np.argmax(f1s))]
    return thresholds


def apply_thresholds(y, p, thresholds):
    sens, f1s, accs = [], [], []
    for i in range(y.shape[1]):
        if not np.any(y[:, i]):
            continue
        pred = (p[:, i] > thresholds[i]).astype(int)
        sens.append(recall_score(y[:, i], pred))
        f1s.append(f1_score(y[:, i], pred))
        accs.append(accuracy_score(y[:, i], pred))
    return float(np.mean(sens)), float(np.mean(f1s)), float(np.mean(accs))


class _ScaffoldArgs:
    pass


def load_trained_model(exp_dir, run_name):
    ckpt_path = os.path.join(exp_dir, run_name + ".pth.tar")
    if not os.path.exists(ckpt_path):
        return None
    _a = _ScaffoldArgs()
    _a.model_name = "swin_large"
    _a.init = "random"          # scaffold only -- real (fine-tuned) weights loaded below
    _a.pretrained_weights = None
    _a.input_size = INPUT_SIZE
    _a.img_size = INPUT_SIZE
    _a.num_class = 14
    _model = build_classification_model(_a)
    _ckpt = torch.load(ckpt_path, map_location=args.device, weights_only=False)  # our own checkpoint; PyTorch >=2.6 default-change workaround, same as models.py/trainer.py/engine.py
    _state_dict = _ckpt["state_dict"]
    _state_dict = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in _state_dict.items()}
    _model.load_state_dict(_state_dict, strict=True)
    _model.to(args.device)
    _model.eval()
    return _model


gap1_rows = []
gap1_raw = {}  # _run -> arrays/values needed for the plotting cell below
for _tier, _seeds in [("screening", SEEDS_SCREENING), ("headline", SEEDS_HEADLINE)]:
    _exp = f"_protocol1_arkplus_{_tier}"
    _exp_dir = os.path.join(model_path, _exp)
    for _i, _seed in enumerate(_seeds):
        _run = f"{_exp}_run_{_i}"
        _pred_csv = os.path.join(_exp_dir, _run + ".csv")
        _gt_csv = os.path.join(_exp_dir, "gt.csv")
        if not (os.path.exists(_pred_csv) and os.path.exists(_gt_csv)):
            print(f"Skipping {_run} -- no cached test predictions yet (run cell 14 through this experiment first).")
            continue

        _model = load_trained_model(_exp_dir, _run)
        if _model is None:
            print(f"Skipping {_run} -- no checkpoint found at {_exp_dir}/{_run}.pth.tar")
            continue

        print(f"Evaluating {_run} ...")
        _val_loader = DataLoader(dataset_val, batch_size=args.batch_size, shuffle=False, num_workers=2)
        y_val_full, p_val_full = test_model(_model, _val_loader, args)
        y_val = y_val_full.cpu().numpy()[:, test_diseases_5]
        p_val = p_val_full.cpu().numpy()[:, test_diseases_5]

        y_test = read_from_csv(_gt_csv)
        p_test = read_from_csv(_pred_csv)

        # Calibration: fit T on val, report ECE before/after on test
        T = fit_temperature(y_val, p_val)
        p_test_calibrated = _sigmoid(_logit(p_test) / T)
        ece_before, _ = meanECE(y_test, p_test)
        ece_after, _ = meanECE(y_test, p_test_calibrated)

        # Three threshold policies, all fit on VAL, applied once to TEST -- computed here (before
        # gap1_raw is stored) so the plotting cell below can reuse them without redoing inference.
        thr_050 = np.full(len(test_diseases_5), 0.5)
        thr_youden = per_label_thresholds(y_val, p_val, mode="youden")
        thr_f1 = per_label_thresholds(y_val, p_val, mode="f1")

        gap1_raw[_run] = {
            "tier": _tier, "seed": _seed,
            "y_test": y_test, "p_test": p_test, "p_test_calibrated": p_test_calibrated,
            "T": T, "ece_before": float(ece_before), "ece_after": float(ece_after),
            "thresholds": {
                "0.5 default": thr_050,
                "Youden's J": thr_youden,
                "F1-optimal": thr_f1,
            },
        }

        gap1_rows.append({
            "tier": _tier, "seed": _seed, "policy": "(calibration)",
            "sensitivity": None, "F1": None, "accuracy": None,
            "ECE_before": round(float(ece_before), 4), "ECE_after": round(float(ece_after), 4),
            "T": round(float(T), 3),
        })

        for _policy_name, _thr in [
            ("0.5 default", thr_050),
            ("Youden's J (val-tuned)", thr_youden),
            ("F1-optimal (val-tuned)", thr_f1),
        ]:
            sens, f1, acc = apply_thresholds(y_test, p_test, _thr)
            gap1_rows.append({
                "tier": _tier, "seed": _seed, "policy": _policy_name,
                "sensitivity": round(sens, 4), "F1": round(f1, 4), "accuracy": round(acc, 4),
                "ECE_before": None, "ECE_after": None, "T": None,
            })

gap1_df = pd.DataFrame(gap1_rows)
print("\nPer-seed results (not pooled -- see note above on why n=2 shouldn't become an error bar):")
display(gap1_df)

In [ ]:
## Gap #1 plots: reliability diagrams (calibration) + accuracy-vs-sensitivity by threshold policy
##
## The "accuracy hides a sensitivity deficit" argument only lands as a picture -- these two
## charts are the actual deliverable from Experiment Plan v3 SS5.2(b) / the earlier gap-analysis
## plan doc ("reliability diagrams + ECE table + threshold-sensitivity analysis"), the table cell
## above only gave the numbers. Headline-tier runs only (the final, reported result); screening
## stays table-only since it's just the cheap ranking pass, not something worth a figure.

import matplotlib.pyplot as plt
import numpy as np

# Colorblind-checked pair (OKLab CVD-simulated deltaE, computed not eyeballed -- same pair used
# in the project status dashboard): teal for accuracy, violet for sensitivity.
_C_ACC = "#0DA0BE"
_C_SENS = "#8D55AA"
_C_MUTED = "#5C6773"

_headline_runs = {k: v for k, v in gap1_raw.items() if v["tier"] == "headline"}
if not _headline_runs:
    print("No headline runs with data yet -- nothing to plot. Run cell 14 through the headline tier first.")
else:
    _n = len(_headline_runs)
    fig, axes = plt.subplots(2, _n, figsize=(5.5 * _n, 9))
    if _n == 1:
        axes = axes.reshape(2, 1)

    for _col, (_run, _d) in enumerate(_headline_runs.items()):
        # --- Row 1: reliability diagram (pooled across the 5 competition classes) ---
        ax = axes[0, _col]
        _bins = np.linspace(0, 1, 11)
        for _label, _p, _color, _style in [
            ("before calibration", _d["p_test"], _C_MUTED, "--"),
            (f"after (T={_d['T']:.2f})", _d["p_test_calibrated"], _C_SENS, "-"),
        ]:
            _y_flat, _p_flat = _d["y_test"].ravel(), _p.ravel()
            _bin_ids = np.clip(np.digitize(_p_flat, _bins) - 1, 0, 9)
            _xs, _ys = [], []
            for _b in range(10):
                _mask = _bin_ids == _b
                if _mask.sum() == 0:
                    continue
                _xs.append(_p_flat[_mask].mean())
                _ys.append(_y_flat[_mask].mean())
            ax.plot(_xs, _ys, _style, color=_color, linewidth=2, marker="o", markersize=4, label=_label)
        ax.plot([0, 1], [0, 1], ":", color=_C_MUTED, linewidth=1, alpha=0.6, label="perfect calibration")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Empirical positive rate")
        ax.set_title(f"{_run}\nECE {_d['ece_before']:.4f} -> {_d['ece_after']:.4f}", fontsize=10)
        ax.legend(fontsize=8, loc="upper left")
        ax.spines[["top", "right"]].set_visible(False)

        # --- Row 2: accuracy vs sensitivity, by threshold policy ---
        ax2 = axes[1, _col]
        _policies = list(_d["thresholds"].keys())
        _sens_vals, _acc_vals = [], []
        for _pname, _thr in _d["thresholds"].items():
            _s, _, _a = apply_thresholds(_d["y_test"], _d["p_test"], _thr)
            _sens_vals.append(_s); _acc_vals.append(_a)
        _x = np.arange(len(_policies))
        _w = 0.35
        _b1 = ax2.bar(_x - _w / 2, _acc_vals, _w, label="Accuracy", color=_C_ACC)
        _b2 = ax2.bar(_x + _w / 2, _sens_vals, _w, label="Sensitivity", color=_C_SENS)
        for _bars in (_b1, _b2):
            ax2.bar_label(_bars, fmt="%.3f", fontsize=8, padding=2)
        ax2.set_xticks(_x); ax2.set_xticklabels(_policies, fontsize=8)
        ax2.set_ylim(0, 1.05)
        ax2.set_ylabel("Score")
        ax2.set_title(f"{_run}: operating point by threshold policy", fontsize=10)
        ax2.legend(fontsize=8)
        ax2.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()